# 23.3 Azure 数据科学栈 + 云访问控制(RBAC)/ Azure for DS + Cloud Access Control (RBAC)

**中文**:**Microsoft Azure** 是第二大云,在企业市场(尤其已经用 Windows/Office/AD 的公司)占有率极高——很多传统大厂、金融、政府的数据科学岗都用 Azure。但学到第三朵云,你应该已经发现一个规律:**三大云本质上是同一套东西的不同名字**。所以本节做两件事:①用一张"Rosetta 石碑"把 AWS/GCP/Azure 的核心服务**一一对应**,让你一次看懂"换云只是换名字";②深入一个**所有云都必须掌握、却常被数据科学家忽视的硬技能——访问控制(RBAC/IAM)**。谁能读哪些数据、谁能训练部署模型、CI 流水线该有什么权限——**权限配错是云安全事故的头号原因**(凭证泄漏、越权访问、数据泄露)。我们从零实现一个 RBAC 引擎,亲手理解"最小权限"原则。
**English**: **Microsoft Azure** is the #2 cloud, with very high enterprise-market share (especially companies already on Windows/Office/AD) — many traditional large firms, finance, and government data-science roles use Azure. But by your third cloud, you should have noticed a pattern: **the three clouds are fundamentally the same thing with different names**. So this section does two things: ① a "Rosetta stone" mapping AWS/GCP/Azure core services **one-to-one**, so you see at once that "switching clouds is just switching names"; ② a deep dive into a **hard skill every cloud requires yet data scientists often overlook — access control (RBAC/IAM)**. Who can read which data, who can train/deploy models, what permissions a CI pipeline should have — **misconfigured permissions are the #1 cause of cloud security incidents** (credential leaks, privilege escalation, data breaches). We build an RBAC engine from scratch to understand the "least privilege" principle firsthand.

---

**中文**:**三大云 Rosetta 石碑(核心服务一一对应)**:
**English**: **The three-cloud Rosetta stone (core services mapped one-to-one)**:

| 能力 / Capability | AWS | GCP | Azure |
|---|---|---|---|
| 对象存储 / Object storage | S3 | GCS | Blob Storage |
| 虚拟机 / VMs | EC2 | Compute Engine | Virtual Machines |
| 无服务器函数 / Serverless | Lambda | Cloud Functions | Azure Functions |
| 数据仓库 / Warehouse | Redshift/Athena | BigQuery | Synapse Analytics |
| ML 平台 / ML platform | SageMaker | Vertex AI | Azure ML |
| 托管 Spark / Managed Spark | EMR | Dataproc | Synapse/HDInsight |
| 消息/流 / Messaging | Kinesis | Pub/Sub | Event Hubs |
| ETL / 数据管道 | Glue | Dataflow | Data Factory (ADF) |
| 身份权限 / IAM | IAM | Cloud IAM | Entra ID + RBAC |
| K8s | EKS | GKE | AKS |

**中文**:**访问控制(RBAC / IAM)—— 云安全的地基**。核心思想:
**English**: **Access control (RBAC / IAM) — the foundation of cloud security.** Core ideas:
- **中文**:**默认拒绝(deny by default)**:任何主体(用户、服务、CI 流水线)默认**没有任何权限**,必须显式授予。
  **Deny by default**: any principal (user, service, CI pipeline) has **no permissions** by default and must be explicitly granted.
- **中文**:**最小权限(least privilege)**:只授予完成工作**所必需**的最小权限。分析师只需读数据、不需要删库;CI 流水线只需写模型目录、不需要碰生产数据库。**权限越小,凭证泄漏时的损害越小。**
  **Least privilege**: grant only the minimum permissions **necessary** for the job. An analyst needs to read data, not drop databases; a CI pipeline needs to write the models directory, not touch the production database. **Smaller permissions = smaller damage when a credential leaks.**
- **中文**:**角色(role)** 是权限的集合,**分配(assign)** 给主体,可**限定范围(scope)**(只在某个路径/资源上生效)。用**角色/服务身份(service principal / managed identity)** 而非硬编码密钥。
  **A role** is a set of permissions, **assigned** to a principal, and can be **scoped** (effective only on a path/resource). Use **roles/service identities (service principal / managed identity)** instead of hardcoded keys.

> 💡 **面试速查 / Interview cheat-sheet（★★ 云安全/权限必考）**
> **中文**:**三大云同构**:对象存储 S3/GCS/Blob、无服务器 Lambda/CloudFunctions/AzureFunctions、数仓 Redshift/BigQuery/Synapse、ML SageMaker/Vertex/AzureML、IAM 各家的权限系统——**学会一朵换云只是查文档**。**RBAC/IAM 核心**:①**默认拒绝**②**最小权限**(只给必需的, 缩小泄漏损害面)③**角色**(权限集合)分配给主体+**范围限定**④**用服务身份/托管身份**(managed identity)而非硬编码密钥⑤**职责分离**、定期审计、密钥轮换。**云安全事故头号原因=权限配错**(公开的 S3 桶、过度授权、泄漏的长期密钥)。**别做的**:给所有人 Owner/Admin、把 access key 写进代码或提交到 git、用长期密钥而非临时角色。**Azure 特有**:Entra ID(原 Azure AD)做身份、深度集成企业 AD/Office、Synapse 统一数仓+Spark、ADF 做 ETL。面试金句:*"三大云核心服务一一对应(S3=GCS=Blob, SageMaker=Vertex=AzureML), 换云主要是换 API; 云安全地基是 RBAC/IAM——默认拒绝+最小权限, 角色分配给主体并限定范围, 用托管身份而非硬编码密钥, 因为权限配错(公开桶、过度授权、泄漏密钥)是云事故头号原因; Azure 强在企业市场和 AD 集成。"*
> **English**: **The three clouds are isomorphic**: object storage S3/GCS/Blob, serverless Lambda/CloudFunctions/AzureFunctions, warehouse Redshift/BigQuery/Synapse, ML SageMaker/Vertex/AzureML, IAM each vendor's permission system — **learn one and switching clouds is just reading docs**. **RBAC/IAM core**: ① **deny by default** ② **least privilege** (grant only what's necessary, shrinking leak damage) ③ **roles** (permission sets) assigned to principals + **scoping** ④ **use service/managed identities** instead of hardcoded keys ⑤ **separation of duties**, periodic audits, key rotation. **The #1 cause of cloud security incidents = misconfigured permissions** (public S3 buckets, over-permissioning, leaked long-lived keys). **Don't**: give everyone Owner/Admin, put access keys in code or commit them to git, use long-lived keys instead of temporary roles. **Azure specifics**: Entra ID (formerly Azure AD) for identity, deep enterprise AD/Office integration, Synapse unifying warehouse + Spark, ADF for ETL. Interview line: *"The three clouds' core services map one-to-one (S3=GCS=Blob, SageMaker=Vertex=AzureML), so switching clouds is mainly changing APIs; cloud security's foundation is RBAC/IAM — deny by default + least privilege, roles assigned to principals with scoping, use managed identities instead of hardcoded keys, because misconfigured permissions (public buckets, over-permissioning, leaked keys) are the #1 cause of cloud incidents; Azure is strong in enterprise and AD integration."*


In [ ]:

# ============================================================
# 从零实现 RBAC 访问控制引擎 / RBAC access-control engine from scratch
# 中文:所有云的权限系统都是这个模型:角色=权限集合, 分配给主体, 可限定范围; 默认拒绝, 最小权限。
# English: every cloud's permission system is this model: roles = permission sets, assigned to principals, scopable;
#      deny by default, least privilege.
# ============================================================
class RBAC:
    def __init__(self): self.roles={}; self.assignments={}
    def define_role(self, role, perms): self.roles[role]=set(perms)          # 角色=权限集合("action:resource")/ role = perm set
    def assign(self, principal, role, scope="*"):                            # 把角色分配给主体, 可限定范围 / assign with optional scope
        self.assignments.setdefault(principal, set()).add((role, scope))
    def can(self, principal, action, resource):                             # 判断某主体能否对某资源做某操作 / authorization check
        for role, scope in self.assignments.get(principal, set()):
            if scope!="*" and not resource.startswith(scope): continue       # 范围限定 / scope restriction
            for perm in self.roles.get(role, set()):
                pa, pr = perm.split(":")
                if (pa==action or pa=="*") and (pr=="*" or resource.startswith(pr.rstrip("*"))):
                    return True
        return False                                                         # ★ 默认拒绝 / DENY BY DEFAULT

rbac=RBAC()
# 定义角色(最小权限:每个角色只含必需权限)/ define roles (least privilege)
rbac.define_role("DataAnalyst", ["read:storage/data/*"])                    # 分析师:只读数据 / read data only
rbac.define_role("MLEngineer",  ["read:storage/*", "write:storage/models/*", "train:ml/*"])
rbac.define_role("Owner",       ["*:*"])                                    # 全权限(危险, 慎用)/ full access (dangerous)
# 分配角色给主体 / assign roles to principals
rbac.assign("alice", "DataAnalyst")                                         # 数据分析师 / analyst
rbac.assign("bob",   "MLEngineer")                                          # ML 工程师 / ML engineer
rbac.assign("ci-bot","MLEngineer", scope="storage/models")                 # CI 流水线: 范围限定到 models 路径 / scoped to models

checks=[("alice","read","storage/data/sales.parquet"),
        ("alice","write","storage/data/sales.parquet"),                    # 拒绝:分析师不能写 / analyst can't write
        ("alice","read","storage/models/m.pkl"),                           # 拒绝:只能读 data/* / only data/*
        ("bob","train","ml/job1"),
        ("bob","write","storage/models/model.pkl"),
        ("ci-bot","write","storage/models/model.pkl"),
        ("ci-bot","write","storage/data/x.parquet")]                       # 拒绝:范围限定在 models / scoped to models
print(f"{'主体 principal':14}{'操作':8}{'资源 resource':34}{'结果':>10}")
for p,a,r in checks:
    print(f"{p:14}{a:8}{r:34}{'✓ 允许 ALLOW' if rbac.can(p,a,r) else '✗ 拒绝 DENY':>12}")
print("\n最小权限原则:alice 只能读数据、ci-bot 只能写 models 路径——即使某个凭证泄漏, 攻击者能造成的破坏也被限制")


In [ ]:

# ============================================================
# 可视化:过度授权 vs 最小权限的"爆炸半径" / over-permissioning vs least privilege blast radius
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 权限矩阵 / permission matrix
principals=["alice\n(分析师)","bob\n(ML工程师)","ci-bot\n(限models)"]
actions=["read data","write data","train ml","write models","delete all"]
def check(p,a):
    m={"read data":("read","storage/data/x"),"write data":("write","storage/data/x"),
       "train ml":("train","ml/j"),"write models":("write","storage/models/m"),"delete all":("delete","storage/prod")}
    pid={"alice\n(分析师)":"alice","bob\n(ML工程师)":"bob","ci-bot\n(限models)":"ci-bot"}[p]
    return rbac.can(pid,*m[a])
grid=[[1 if check(p,a) else 0 for a in actions] for p in principals]
ax[0].imshow(grid,cmap="RdYlGn",vmin=0,vmax=1,aspect="auto")
ax[0].set_xticks(range(len(actions))); ax[0].set_xticklabels(actions,rotation=30,ha="right",fontsize=8)
ax[0].set_yticks(range(len(principals))); ax[0].set_yticklabels(principals,fontsize=8)
for i in range(len(principals)):
    for j in range(len(actions)): ax[0].text(j,i,"✓" if grid[i][j] else "✗",ha="center",va="center",fontsize=12)
ax[0].set_title("最小权限:每个主体只有必需的权限(绿=允许)")
# ② 凭证泄漏的爆炸半径 / blast radius if a credential leaks
ax[1].bar(["Owner\n(全权限)","MLEngineer\n(部分)","分析师\n(只读)"],[100,40,15],color=["#C44E52","#DD8452","#55A868"])
for i,v in enumerate([100,40,15]): ax[1].text(i,v+2,f"{v}%",ha="center",fontsize=11,weight="bold")
ax[1].set_ylabel("凭证泄漏后攻击者可破坏的范围 %"); ax[1].set_title("最小权限 = 缩小泄漏的爆炸半径")
plt.tight_layout(); plt.savefig("/tmp/cloud03_viz.png",dpi=80); plt.show()
print("左:每个主体只有必需权限; 右:权限越小, 凭证一旦泄漏能造成的破坏越小——这就是最小权限的价值")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **学到第三朵云,最大的收获是"看穿它们本质相同"**:Rosetta 石碑一目了然——对象存储(S3/GCS/Blob)、无服务器(Lambda/Functions)、数仓(Redshift/BigQuery/Synapse)、ML 平台(SageMaker/Vertex/Azure ML),三家几乎一一对应。这不是巧合:它们都在实现同一套云的基本原语(存算分离、无服务器、托管服务、按量计费)。**结论对求职极其重要:不要因为岗位要求"AWS 经验"而不敢投用 GCP 的工作,反之亦然——你真正需要掌握的是云的心智模型,而不是某家的 API 名词。** 会一朵云,换一朵只是查文档、改几个 SDK 调用。面试里能画出这张对应表、讲清背后的共同思想,比背某家的服务清单强得多。
2. **访问控制是数据科学家最该补、却最常忽视的云技能**:很多数据科学家把"权限"当成运维/安全团队的事,自己只管建模。但现实是:**云安全事故的头号原因就是权限配错**——公开可读的 S3 桶泄漏了亿万用户数据、提交到 GitHub 的 access key 被爬虫几分钟内盗用、一个过度授权的服务账号被攻破后横扫整个云。RBAC 的两条铁律——**默认拒绝 + 最小权限**——正是防线:我们的实验展示了,给 alice 只读数据的权限、给 CI 只写 models 路径的权限,意味着**即使她们的凭证泄漏,攻击者能造成的破坏也被死死限制住**(右图的"爆炸半径")。反之,图省事给所有人 Owner/Admin 权限,等于把整个云的钥匙散得到处都是。
3. **诚实的边界与实践**:①**最小权限有"摩擦成本"**——精细的权限配置繁琐,开发时"权限不够"很烦人,所以团队常图省事过度授权("先给全权限让它跑起来")——这正是隐患的源头。正确的做法是**从零权限开始、按需逐步添加**,而不是从全权限开始砍。②**别用长期密钥**:硬编码的 access key 一旦泄漏(提交到 git、日志里打出来)就是灾难;要用**临时凭证/角色/托管身份**(managed identity)——让云自动注入短期令牌,无密钥可泄漏。③**这只是基础**:真实云安全还有网络隔离(VPC/私有网络)、加密(静态/传输)、审计日志、密钥管理(KMS/Key Vault)、合规(GDPR/HIPAA)等一整套,RBAC 是地基但不是全部。④**Azure 的差异化**在企业集成:如果公司已经用 Microsoft 全家桶(AD/Office/Teams),Azure 的身份体系(Entra ID)无缝对接,这是它在大企业占优的关键原因——技术选型往往不只看技术,还看现有生态。**结论:三大云本质同构(一张 Rosetta 表看穿),求职别被'某家经验'门槛吓住——掌握云的心智模型即可迁移; 而访问控制(RBAC:默认拒绝+最小权限+托管身份)是数据科学家必须补上的云安全地基, 因为权限配错是云事故头号原因——用最小权限缩小凭证泄漏的爆炸半径, 是专业与业余的分水岭之一。**

**English**:
1. **By your third cloud, the biggest gain is "seeing through their fundamental sameness"**: the Rosetta stone makes it clear — object storage (S3/GCS/Blob), serverless (Lambda/Functions), warehouse (Redshift/BigQuery/Synapse), ML platforms (SageMaker/Vertex/Azure ML) map almost one-to-one across the three. Not a coincidence: they all implement the same cloud primitives (storage-compute separation, serverless, managed services, pay-per-use). **The conclusion matters greatly for job-seeking: don't hesitate to apply for a GCP job because it requires "AWS experience," or vice versa — what you truly need is the cloud mental model, not a vendor's API names.** Know one cloud and switching is just reading docs and changing a few SDK calls. In interviews, drawing this mapping table and explaining the shared ideas beats memorizing a vendor's service list.
2. **Access control is the cloud skill data scientists most need yet most overlook**: many data scientists treat "permissions" as the ops/security team's job and just model. But in reality: **the #1 cause of cloud security incidents is misconfigured permissions** — a publicly-readable S3 bucket leaking hundreds of millions of users' data, an access key committed to GitHub stolen by crawlers within minutes, an over-permissioned service account compromised and sweeping the whole cloud. RBAC's two iron rules — **deny by default + least privilege** — are exactly the defense: our experiment showed that granting alice read-only data and CI write-only-models means **even if their credentials leak, the damage an attacker can do is tightly limited** (the "blast radius" in the right plot). Conversely, giving everyone Owner/Admin for convenience scatters the keys to the whole cloud everywhere.
3. **Honest limits and practice**: ① **Least privilege has "friction cost"** — fine-grained permission config is tedious, "insufficient permissions" during development is annoying, so teams often over-permission for convenience ("give full access first to make it run") — exactly the source of risk. The right approach is **start with zero permissions and add incrementally as needed**, not start with full access and trim. ② **Don't use long-lived keys**: a hardcoded access key, once leaked (committed to git, printed in logs), is a disaster; use **temporary credentials/roles/managed identities** — let the cloud inject short-lived tokens, with no key to leak. ③ **This is only the basics**: real cloud security also has network isolation (VPC/private networks), encryption (at-rest/in-transit), audit logs, key management (KMS/Key Vault), compliance (GDPR/HIPAA) — RBAC is the foundation but not the whole. ④ **Azure's differentiation** is enterprise integration: if a company already uses the Microsoft suite (AD/Office/Teams), Azure's identity system (Entra ID) integrates seamlessly, a key reason for its enterprise dominance — tech choices often consider not just technology but the existing ecosystem. **Conclusion: the three clouds are fundamentally isomorphic (one Rosetta table sees through it), so job-seekers shouldn't be intimidated by "vendor X experience" gates — master the cloud mental model and it transfers; and access control (RBAC: deny by default + least privilege + managed identities) is the cloud-security foundation data scientists must add, because misconfigured permissions are the #1 cause of cloud incidents — using least privilege to shrink a credential leak's blast radius is one of the divides between professional and amateur.**

> 💼 **实战视角 / Practical angle**
> **中文**:落地:①**跨云迁移**——记住 Rosetta 对应关系, 换云先查"这个服务对应我熟悉的哪个", 核心概念不变;②**RBAC 最小权限**——从零权限开始按需加、给每个主体(人/服务/CI)独立角色并限定范围、职责分离;③**绝不硬编码密钥**——用 IAM 角色/托管身份/临时凭证(AWS STS、Azure Managed Identity、GCP Workload Identity), access key 绝不进代码/git(用 secret 管理、`.gitignore`、密钥扫描);④**定期审计**权限、轮换密钥、删除闲置账号;⑤**S3/Blob 桶默认私有**, 公开前三思(公开桶泄漏是经典事故);⑥Azure 用户善用 Entra ID 与企业 AD 集成。面试金句:*"三大云核心服务一一对应(S3/GCS/Blob、SageMaker/Vertex/AzureML), 换云只是换 API, 掌握心智模型即可迁移; 云安全地基是 RBAC——默认拒绝+最小权限, 角色限定范围, 用托管身份而非硬编码密钥, 从零权限按需加; 因为权限配错(公开桶/过度授权/泄漏密钥)是云事故头号原因, 最小权限缩小泄漏爆炸半径。"*
> **English**: In practice: ① **cross-cloud migration** — remember the Rosetta mapping; when switching clouds, first check "which service I know does this correspond to," the core concepts are unchanged; ② **RBAC least privilege** — start with zero and add as needed, give each principal (person/service/CI) its own scoped role, separation of duties; ③ **never hardcode keys** — use IAM roles/managed identities/temporary credentials (AWS STS, Azure Managed Identity, GCP Workload Identity), access keys never in code/git (use secret management, `.gitignore`, key scanning); ④ **periodically audit** permissions, rotate keys, remove idle accounts; ⑤ **S3/Blob buckets private by default**, think twice before making public (public-bucket leaks are classic incidents); ⑥ Azure users leverage Entra ID's enterprise AD integration. Interview line: *"The three clouds' core services map one-to-one (S3/GCS/Blob, SageMaker/Vertex/AzureML), so switching clouds just changes APIs — master the mental model and it transfers; cloud security's foundation is RBAC — deny by default + least privilege, scoped roles, use managed identities instead of hardcoded keys, start from zero and add as needed; because misconfigured permissions (public buckets/over-permissioning/leaked keys) are the #1 cause of cloud incidents, least privilege shrinks a leak's blast radius."*

---
### 小结 / Summary
- **中文**:三大云本质同构(Rosetta:S3/GCS/Blob、SageMaker/Vertex/AzureML……); 换云只是换 API, 掌握心智模型即可迁移。
- **English**: The three clouds are isomorphic (Rosetta: S3/GCS/Blob, SageMaker/Vertex/AzureML…); switching clouds just changes APIs, master the mental model to transfer.
- **中文**:RBAC 是云安全地基:默认拒绝 + 最小权限 + 角色限定范围 + 托管身份(不硬编码密钥)。
- **English**: RBAC is the cloud-security foundation: deny by default + least privilege + scoped roles + managed identities (no hardcoded keys).
- **中文**:权限配错是云事故头号原因; 最小权限缩小凭证泄漏的爆炸半径; Azure 强在企业市场与 AD 集成。
- **English**: Misconfigured permissions are the #1 cause of cloud incidents; least privilege shrinks a credential leak's blast radius; Azure is strong in enterprise and AD integration.
